# Answers to Assignment 1

### Problem (unicode)
1. ```chr(0) = '\x00'```
2. ```__repr__() = "'\\x00'"``` ,print prints nothing
3. The charecter ```\0``` is the end of string so the print doesnt print the escaped character it is hidden ig

In [ ]:
chr(0)

In [ ]:
print(chr(0))

In [ ]:
"this is a test" + chr(0) + "string"

In [ ]:
print("this is a test" + chr(0) + "string")

### Problem (unicode2)

1. Is it sparsity? Bigger encodings just waste token representations and increase the individual thing
2. ```こんにちは``` wont be decoded. The problem is that sometimes decode needs more than just one byte to get
3. ```b1000_1000_1000``` is an invalid byte sequence

In [ ]:
len("this is a cool string".encode('utf-8'))

In [ ]:
len("this is a cool string".encode('utf-16'))

In [ ]:
len("this is a cool string".encode('utf-32'))

In [ ]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [ ]:
decode_utf8_bytes_to_str_wrong("こんにちは")

### Problem (train_bpe_tinystories)
1. It took 265 MB and 2208.729ms to train the tokenizer. (26s for the chunking with 2.4 GB of collective memory on 24 cores). The longest words are  ```[accomplishment,disappointment, responsibility]``` which make senseish ig
2. ```cupsert`` My helper method to add and remove in the heap took a lot of time. The build_list helper took a major chunk in the start and then there were a lot of removals at the start accounting for 24% of the time and build list (initial heap initialization) accounting for 32% of the time

In [2]:
from cs336_basics.train.tokenizer import train_tokenizer
from cs336_basics.pretokenize import count_pretokens_mapper, count_pretokens_reducer, multiproc_pretokens
from functools import partial
from collections import Counter
import os
from cs336_basics.perf import pyspy_profile, PeakMoniter
import pickle
import time

TINY_STORIES = "data/TinyStoriesV2-GPT4-train.txt"
OWL = "data/owt_train.txt"
NPROC = os.cpu_count()
special_tokens = ["<|endoftext|>"]

Tiny Stories Profiling: 

In [2]:
print("Running count pretokenize profiling for TINY_STORIES:")
tiny_stories_pretoken_counter = Counter()
monitor = PeakMoniter()
monitor.start()
with pyspy_profile(output=f"profiles/tiny_stories_count_pretokens.json", rate=100):
    multiproc_pretokens(
        TINY_STORIES,
        partial(count_pretokens_mapper, end_token="<|endoftext|>"),
        count_pretokens_reducer,
        tiny_stories_pretoken_counter,
        numprocs=NPROC
    )

print(f"peak (rss,pss) for pretoken counting TINY_STORIES {monitor.stop()} MB")

Running count pretokenize profiling for TINY_STORIES:
Sucessfully ran the py-spy
py-spy> Sampling process 100 times a second. Press Control-C to exit.



pretokenizing in parallel: 100%|██████████| 96/96 [00:26<00:00,  3.69it/s]


Sending SIGINT to py-spy
Waiting for Proc to py-spy to quit

py-spy> Stopped sampling because Control-C pressed
py-spy> Wrote speedscope file to 'profiles/tiny_stories_count_pretokens.json'. Samples: 62024 Errors: 0
py-spy> Visit https://www.speedscope.app/ to view
py-spy quit
peak (rss,pss) for pretoken counting TINY_STORIES (2741.067776, 2741.067776) MB


In [3]:
print("Running train_tokenizer profiling for TINY_STORIES:")
monitor = PeakMoniter()
monitor.start()
start = time.perf_counter()
with pyspy_profile(output=f"profiles/tiny_stories_train_tokenizer.json", rate=100):
    v, m = train_tokenizer(10_000, ["<|endoftext|>"],tiny_stories_pretoken_counter)

print(f"training took {(time.perf_counter() - start) * 1000:.3f}ms")
print(f"peak (rss,pss) for training tokenizer on TINY_STORIES {monitor.stop()} MB")

with open("models/tiny_stories_tokenizer.pkl", "wb") as file:
    pickle.dump((v,m), file)

Running train_tokenizer profiling for TINY_STORIES:
Sucessfully ran the py-spy
py-spy> Sampling process 100 times a second. Press Control-C to exit.



BPE Merges: 100%|██████████| 9743/9743 [00:01<00:00, 5896.23it/s] 

Sending SIGINT to py-spy
Waiting for Proc to py-spy to quit

py-spy> Stopped sampling because Control-C pressed
py-spy> Wrote speedscope file to 'profiles/tiny_stories_train_tokenizer.json'. Samples: 170 Errors: 0
py-spy> Visit https://www.speedscope.app/ to view
py-spy quit
training took 2208.729ms
peak (rss,pss) for training tokenizer on TINY_STORIES (265.58464, 265.58464) MB


In [4]:
vocab = sorted(v.values(), key=len)
print("\n".join([v.decode(encoding='utf-8') for v in vocab if len(v) == len(vocab[-1])]))

 accomplishment
 disappointment
 responsibility


### Problem (train_bpe_expts_owt)
1. It took 14.4 GB and 368.778s to train the tokenizer. (132s for the chunking with 10.3 GB of collective memory on 24 cores). The longest are  ```[ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ, ----------------------------------------------------------------]``` which probably come due to the fact that our vocabulary can fit much longer tokens
2. ```removals``` followed by a stream of adds was the trend for this. Tokenizer size was 732K roughly 3.3 times which follows perfectly from the vocab sizes. 

In [11]:
print("Running count pretokenize profiling for OWL:")
owl_pretoken_counter = Counter()
monitor = PeakMoniter()
monitor.start()
start = time.perf_counter()
with pyspy_profile(output=f"profiles/owl_count_pretokens.json", rate=100):
    multiproc_pretokens(
        OWL,
        partial(count_pretokens_mapper, end_token="<|endoftext|>"),
        count_pretokens_reducer,
        owl_pretoken_counter,
        numprocs=NPROC
    )

print(f"pretokening took {(time.perf_counter() - start):.3f}s")
print(f"peak (rss,pss) for pretoken counting OWL {monitor.stop()} MB")

Running count pretokenize profiling for OWL:
Sucessfully ran the py-spy
py-spy> Sampling process 100 times a second. Press Control-C to exit.



pretokenizing in parallel: 100%|██████████| 96/96 [02:11<00:00,  1.37s/it]


Sending SIGINT to py-spy
Waiting for Proc to py-spy to quit

py-spy> Stopped sampling because Control-C pressed
py-spy> Wrote speedscope file to 'profiles/owl_count_pretokens.json'. Samples: 299275 Errors: 0
py-spy> Visit https://www.speedscope.app/ to view
py-spy quit
pretokening took 132.468s
peak (rss,pss) for pretoken counting OWL (10516.934656, 10516.934656) MB


In [12]:
print("Running train_tokenizer profiling for OWL:")
monitor = PeakMoniter()
monitor.start()
start = time.perf_counter()

with pyspy_profile(output=f"profiles/owl_train_tokenizer.json", rate=100):
    v, m = train_tokenizer(32_000, ["<|endoftext|>"],owl_pretoken_counter)
    
print(f"training took {(time.perf_counter() - start):.3f}s")
print(f"peak (rss,pss) for training tokenizer on OWL {monitor.stop()} MB")
with open("models/owl_tokenizer.pkl", "wb") as file:
    pickle.dump((v,m), file)

Running train_tokenizer profiling for OWL:
Sucessfully ran the py-spy
py-spy> Sampling process 100 times a second. Press Control-C to exit.



BPE Merges: 100%|██████████| 31743/31743 [06:05<00:00, 86.78it/s]  


Sending SIGINT to py-spy
Waiting for Proc to py-spy to quit

py-spy> Stopped sampling because Control-C pressed
py-spy> Wrote speedscope file to 'profiles/owl_train_tokenizer.json'. Samples: 36610 Errors: 0
py-spy> Visit https://www.speedscope.app/ to view
py-spy quit
pretokening took 368.778s
peak (rss,pss) for training tokenizer on OWL (14715.293696, 14715.285504) MB


In [13]:
vocab = sorted(v.values(), key=len)
print("\n".join([v.decode(encoding='utf-8') for v in vocab if len(v) == len(vocab[-1])]))

ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ
----------------------------------------------------------------


### Problem (tokenizer_experiments)
1. The compression ratio is about 4 for all tokenizer with owl performing the best on its own set with 4.3
2. Owl performs well reaching 3.9 on tiny token but tiny tok fails at 3.1 which shows that tiny tokenizers vocab isn't big enough to compress owl's tokens
3. Throughput of my tokenizer is 13 MB/s. Pile Dataset would take 18 Hours. Note that this is using 24 cores
4. uint16 is an apt choice because it is the nearest power of 2 which allows for the tokenizers tokens to fit (technically 2^15) fits but well we like even bits :)

In [1]:
import math
print(825 * (2 ** 10) / 13 / 60/60)
print(math.ceil(math.log2(32_000)))

18.05128205128205
15


In [2]:
from cs336_basics.tokenizer import BPETokenizer

owl_tok = BPETokenizer.from_file('models/owl_tokenizer.pkl')
tiny_tok = BPETokenizer.from_file('models/tiny_stories_tokenizer.pkl')
TINY_STORIES = "data/TinyStoriesV2-GPT4-train.txt"
OWL = "data/owt_train.txt"


In [3]:
from random import shuffle
with open(TINY_STORIES, "r") as tiny: 
    tiny_10_docs = tiny.read(4096 * 100).split('<|endoftext|>')
    shuffle(tiny_10_docs)
    tiny_10_docs = tiny_10_docs[:10]

with open(OWL, "r") as owl: 
    owl_10_docs = owl.read(4096 * 100).split('<|endoftext|>')
    shuffle(owl_10_docs)
    owl_10_docs = owl_10_docs[:10]


In [4]:
[doc[:100]+'...' for doc in tiny_10_docs][:3]

['\nOne day, a little red car named Zoom went for a ride. Zoom loved to go fast. He would go up and dow...',
 '\nOnce upon a time, in a small town, there lived a boy named Tim. One day, it began to rain. The rain...',
 '\nOne day, a little kid named Tim was going to the park. He was nervous because it was his first time...']

In [5]:
[doc[:100]+'...' for doc in owl_10_docs][:3]

['By Joshua Krause\n\nFor the past two years the families of the victims who died at Sandy Hook have bee...',
 'SASKATOON – One day after Saskatchewan NDP Leader Cam Broten introduced his party’s full platform, S...',
 'The ringleader of the London Bridge atrocity tried to hire a 7.5-ton lorry just hours before the dea...']

In [6]:
import numpy as np
def compression_ratio(tokenizer: BPETokenizer, doc: str): 
    doc_b = doc.encode('utf-8')
    encoded = tokenizer.encode(doc)
    return len(doc_b) / len(encoded)

owl_v_owl = [compression_ratio(owl_tok, owl_doc) for owl_doc in owl_10_docs]
tiny_v_owl = [compression_ratio(tiny_tok, owl_doc) for owl_doc in owl_10_docs]
owl_v_tiny = [compression_ratio(owl_tok, tiny_doc) for tiny_doc in tiny_10_docs]
tiny_v_tiny = [compression_ratio(tiny_tok, tiny_doc) for tiny_doc in tiny_10_docs]

In [7]:
from statistics import mean, stdev

print(f"owl_v_owl  : mean = {mean(owl_v_owl):.3f} stdev = {stdev(owl_v_owl):.3f}")
print(f"owl_v_tiny : mean = {mean(owl_v_tiny):.3f} stdev = {stdev(owl_v_tiny):.3f}")
print(f"tiny_v_owl : mean = {mean(tiny_v_owl):.3f} stdev = {stdev(tiny_v_owl):.3f}")
print(f"tiny_v_tiny: mean = {mean(tiny_v_tiny):.3f} stdev = {stdev(tiny_v_tiny):.3f}")

owl_v_owl  : mean = 4.543 stdev = 0.322
owl_v_tiny : mean = 3.974 stdev = 0.184
tiny_v_owl : mean = 3.238 stdev = 0.360
tiny_v_tiny: mean = 4.057 stdev = 0.171


In [8]:
import time
import os 
from cs336_basics.pretokenize import multiproc_pretokens, SPECIAL_EOT_TOKEN, encode_pretokenize_reducer, encode_pretokenize_mapper
from functools import partial
import numpy as np
from cs336_basics.perf import PeakMoniter, pyspy_profile

NPROC = os.cpu_count()

In [ ]:
start = time.perf_counter()
file = "data/owt_valid.txt"
encoded_file = []

multiproc_pretokens(
    file,
    partial(encode_pretokenize_mapper, tokenizer=owl_tok),
    encode_pretokenize_reducer,
    encoded_file,
    numprocs=NPROC
)

walltime = time.perf_counter() - start
print(f"training took {walltime:.3f}s")


print(f"OWL throughput: { os.path.getsize(file) / walltime / 1e6: .3f} MB / s")

In [ ]:
file_tokenized = np.concatenate([arr for _, arr in sorted(encoded_file, key=lambda x: x[0])])
np.save("data/tokenized/owl_valid_tokens_with_owl.npy", file_tokenized) 

### Problem (transformer_accounting)
1. Architecture counting:
```python
vocab_size = 50_257
max_context = 1_024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 4_288

# Transformer is: token -> [Embedding In] -> num_layers * { [Norm]-> {MHA} -> [Norm] -> SwiGLU} -> [Norm] -> [Linear Emb Out] -> Softmax
# MHA = num_heads * {Wq, Wk, Wv} ->  Wo -> X
# dk = d_model // num_heads, dv = d_model // numheads

dk = d_model // num_heads
dv = d_model // num_heads

MHA = num_heads * (d_model * dk + d_model * dk + d_model * dv) + d_model * (dv * num_heads)
TRANSFORMER = vocab_size * d_model + num_layers * (d_model + MHA + d_model + 3 * d_model * d_ff ) + d_model + d_model * vocab_size
```
This brings to 1640452800 or about 1.64 billion params
2. 
```python
tokens = max_context

# Embedding In -> 0 FLOPS
# Normalization: sqrt(aTa) + a / RMS(a) + a * y_i
#   brings 2 * d_model + d_model + d_model + d_model = 5 d_model flops
#              aTa         sqrt      /          * 
# MHA: [n_heads * ( softmax(xWq Wk.T x.T / sqrt(dk)) Wv x )]  Wo 
#   xWq -- 2 * dmodel * dk


```


In [13]:
vocab_size = 50_257
max_context = 1_024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 4_288

# Transformer is: token -> [Embedding In] -> num_layers * { [Norm]-> {MHA} -> [Norm] -> SwiGLU} -> [Norm] -> [Linear Emb Out] -> Softmax
# MHA = num_heads * {Wq, Wk, Wv} ->  Wo -> X
# dk = d_model // num_heads, dv = d_model // numheads

dk = d_model // num_heads
dv = d_model // num_heads

MHA = num_heads * (d_model * dk + d_model * dk + d_model * dv) + d_model * (dv * num_heads)
TRANSFORMER = vocab_size * d_model + num_layers * (d_model + MHA + d_model + 3 * d_model * d_ff ) + d_model + d_model * vocab_size

In [14]:
TRANSFORMER

1640452800

In [21]:
from cs336_basics.layers import Transformer

t = Transformer(
    vocab_size=vocab_size,
    d_model = d_model,
    num_layers = num_layers,
    num_heads = num_heads,
    d_ff = d_ff
)

sum(p.numel() for p in t.parameters())

1640452800